In [1]:
file = 'balakanda_chapter1.json'
import json
with open(file) as f:
    data = json.load(f)

from llama_index.core import Document
documents = []
for i in data[:]:       
    document = Document(
    text = i['content'],
    metadata={
        "canto": i['chapter'],
        "canto_summary": i['summary'],
        "r_id": str(i['chapter']) + "_" + str(i['prompt_number'])
    },
    excluded_llm_metadata_keys=["r_id"],
    metadata_seperator="::",
    metadata_template="{key}=>{value}",
    text_template="Content: {content}, Extra Chapter context for reference: {metadata_str}\n",
    )
    documents.append(document)
print(len(documents))
print(f"Document Metadata: {documents[-1].metadata}")

[nltk_data] Downloading package punkt_tab to c:\Users\M LAKSHMI
[nltk_data]     NARASIMHA\anaconda3\envs\RAMAYAN\Lib\site-
[nltk_data]     packages\llama_index\core\_static/nltk_cache...
[nltk_data]   Package punkt_tab is already up-to-date!


21
Document Metadata: {'canto': '1', 'canto_summary': '', 'r_id': '1_20'}


In [2]:
from gemini import ramayandata
from tqdm import tqdm
def generate_triplets(documents):
    DEFAULT_KG_TRIPLET_EXTRACT_TMPL_NEW = (
    "Some text from ancient story Ramayan is provided below. Given the text, extract knowledge graph triplets with the following rules:\n"

    "1. Extract triplets with the following patterns:\n"
    "   (charecter, relation, charecter) - Resolve the relations between charecters based on the text given\n"
    "   (charecter, context, event) - Resolve the relations between charecters and the major events in the text given\n"
    "   (charecter, 'is in', Canto) - Mapping for each charecter to canto\n"
    "2. Focus more on the important events and relations between charecters\n"
    "3. Avoid stopwords. The output should only be triplets. Following are few examples for syntax:\n"
    "Example1: ('rama', 'is_son_of', 'dasaratha')\n"
    "Example2: ('bharata', 'is_son_of', 'dasaratha')\n"
    "4. use snake casing for everything in triplets\n"
    "5. for (charecter, relation, charecter) type of triplets, use the following relations:\n"
    "- is (mother/father/brother/sister/wife/husband/son/daughter/siblings/aunt/uncle/niece/nephew/teacher/mentor/related_in_canto_'number') of\n"
    "7. The number of triplets triplets of type (charecter, 'is in', Canto) should not exceed 10, and the number of other two type of triplets should not exceed {max_knowledge_triplets} \n"
    "These triplets are used for RAG, so make sure there is no important data missing\n"
    "Text: {text}\n"
    "Triplets:\n"
    )
    for document in tqdm(documents):
        text = document.get_text()
        metadata_str = document.get_metadata_str()
        data = f"Content: {text}, Extra Chapter context for reference: {metadata_str}\n"
        prompt = DEFAULT_KG_TRIPLET_EXTRACT_TMPL_NEW.format(text=data, max_knowledge_triplets=20)
        triplets = ramayandata(prompt)
        with open('balakanda_chapter_1_triplets.json', 'a') as f:
            key = document.metadata['r_id']
            json.dump({key: triplets}, f, indent=4)
            f.write(",\n")

c:\Users\M LAKSHMI NARASIMHA\anaconda3\envs\RAMAYAN\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
generate_triplets(documents)

100%|██████████| 21/21 [01:24<00:00,  4.01s/it]
